# ⚡ AI Friend: LLM Throughput Benchmark on a Colab GPU

<a href="https://colab.research.google.com/github/PALabs-v1/AI_friend/blob/main/notebooks/ai_friend_llm_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Measures raw Ollama generation throughput (tokens/sec), time-to-first-token,
and VRAM footprint across candidate model sizes on a real GPU -- the
question this answers is **"is the local 3B ceiling a hardware limit or a
Mac-specific one, and what would a bigger model actually cost?"**
-- 3B is a Mac-hardware ceiling, not an architectural one, and a rented GPU
is the project's own plan for anything past it -- this notebook is the
cheap first look before renting anything.

**Standalone by design.** Unlike the other two notebooks, this one does not
clone the repo -- it talks to Ollama's HTTP API directly, so it has nothing
to break if the repo changes shape. It borrows the MEASURED/ESTIMATED/
UNKNOWN provenance discipline from `backend/tools/measure/` (see that
module's `schema.py`) without importing it, for the same reason: no reason
to drag in `app.config` and its dependencies for what's fundamentally five
HTTP calls in a loop.

**What this is not:** it does not run the cognitive pipeline (appraisal,
decision, action, reflection -- six sequential LLM calls per real turn, see
`docs/FUTURE_WORK.md` 4.3) and it is not one of the registered `m11`-`m17`
measurements in `backend/tools/measure/out/` -- those need the full NATS/
Postgres/Neo4j mesh, which does not reliably run inside Colab's own
container (nested Docker there is unsupported, not just inconvenient). If
you fold this notebook's output into the ledger, describe it as what it is:
a standalone raw-generation throughput measurement, not a relabeled m-number.

### Quick instructions
1. Run **Cell 1** (GPU check + install/start Ollama).
2. Edit and run **Cell 2** to pull your candidate models.
3. Run **Cell 3** to benchmark all of them against a fixed prompt set.
4. Run **Cell 4** for the summary table and chart.
5. Run **Cell 5** to download the raw JSON.

In [ ]:
# Cell 1 -- GPU check, install & start Ollama
import subprocess
import time

import httpx

try:
    out = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True)
    print(out.stdout)
except (subprocess.CalledProcessError, FileNotFoundError):
    print(
        "WARNING: no GPU detected -- throughput numbers will describe CPU "
        "inference, not the GPU comparison this notebook is for. Runtime "
        "-> Change runtime type if that wasn't intended."
    )

!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

_ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/content/ollama.log", "a"),
    stderr=subprocess.STDOUT,
)

for _ in range(60):
    try:
        r = httpx.get("http://127.0.0.1:11434/api/tags", timeout=2.0)
        if r.status_code == 200:
            print("Ollama is up.")
            break
    except httpx.HTTPError:
        pass
    time.sleep(1)
else:
    raise RuntimeError(
        "Ollama never came up -- check /content/ollama.log:\n"
        + open("/content/ollama.log").read()[-2000:]
    )

### Cell 2 -- Pull candidate models

Pick a spread that brackets the question you actually have. The default set
brackets the local 3B ceiling on both sides.

In [ ]:
models = "hermes3:8b,qwen2.5:14b,mistral-nemo:12b,llama3.2:3b"  # @param {type:"string"}
MODEL_LIST = [m.strip() for m in models.split(",") if m.strip()]

for m in MODEL_LIST:
    print(f"--- pulling {m} ---")
    !ollama pull {m}

### Cell 3 -- Benchmark

For each model: unload any resident model (`keep_alive: 0`), reload it and
burn one throwaway generation -- the same reset `evals/runner.py`'s
`reset_model_state` performs, because a model's throughput on a "cold" load
measurably differs from one still holding a previous run's KV cache warm.
Then time N fixed prompts with `stream=True`, so time-to-first-token is
measured directly rather than estimated from total latency.

In [ ]:
import json
import subprocess
import time

import httpx

PROMPTS = [
    "I had a really exhausting and overwhelming day today. Can we just talk for a bit?",
    "Remember how you told me last week that you like blues guitar? What was that chord progression you mentioned?",
    "What is the weather like in your mind right now? Give me a warm, poetic 2-sentence answer.",
    "If we could take a walk anywhere right now, where would we go and what would we talk about?",
    "Analyze emotional state: I cannot take this anymore, everything is piling up. Output valid JSON: valence, arousal, dominance, inferred_need.",
]
NUM_PREDICT = 192  # matches evals/schema.py's RunOptions default, for comparability
CLIENT_TIMEOUT = 120.0


def _vram_used_mb():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            capture_output=True,
            text=True,
            check=True,
        )
        return float(out.stdout.strip().splitlines()[0])
    except (subprocess.CalledProcessError, FileNotFoundError, ValueError):
        return None


def _reset_model(client, model):
    # Unload whatever is resident, then reload and burn one throwaway
    # generation -- mirrors evals/runner.py::reset_model_state's finding that
    # "freshly loaded" and "holding a previous run's residue" are different
    # starting points for timing, not just for output text.
    client.post(
        "http://127.0.0.1:11434/api/generate",
        json={"model": model, "prompt": "", "keep_alive": 0},
        timeout=CLIENT_TIMEOUT,
    )
    time.sleep(1)
    client.post(
        "http://127.0.0.1:11434/api/generate",
        json={
            "model": model,
            "prompt": "warm up",
            "stream": False,
            "options": {"num_predict": 8},
        },
        timeout=CLIENT_TIMEOUT,
    )


def _timed_generate(client, model, prompt):
    t0 = time.monotonic()
    first_token_t = None
    token_count = 0
    with client.stream(
        "POST",
        "http://127.0.0.1:11434/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": True,
            "options": {"num_predict": NUM_PREDICT, "temperature": 0.0, "seed": 42},
        },
        timeout=CLIENT_TIMEOUT,
    ) as resp:
        for line in resp.iter_lines():
            if not line:
                continue
            chunk = json.loads(line)
            if chunk.get("response") and first_token_t is None:
                first_token_t = time.monotonic()
            token_count += 1
            if chunk.get("done"):
                total_t = time.monotonic()
                eval_count = chunk.get("eval_count", token_count)
                eval_duration_ns = chunk.get("eval_duration", 0)
                break
    ttft_s = (first_token_t - t0) if first_token_t else None
    tokens_per_sec = (
        (eval_count / (eval_duration_ns / 1e9)) if eval_duration_ns else None
    )
    return {
        "prompt": prompt,
        "ttft_s": ttft_s,
        "total_s": total_t - t0,
        "eval_count": eval_count,
        "tokens_per_sec": tokens_per_sec,
    }


results = {}
with httpx.Client() as client:
    for model in MODEL_LIST:
        print(f"=== {model} ===")
        _reset_model(client, model)
        vram_before = _vram_used_mb()
        runs = [_timed_generate(client, model, p) for p in PROMPTS]
        vram_after = _vram_used_mb()
        results[model] = {
            "runs": runs,
            "vram_used_mb": vram_after,
            "vram_delta_mb": (
                (vram_after - vram_before)
                if vram_before is not None and vram_after is not None
                else None
            ),
        }
        mean_tps = sum(r["tokens_per_sec"] for r in runs if r["tokens_per_sec"]) / len(
            runs
        )
        mean_ttft = sum(r["ttft_s"] for r in runs if r["ttft_s"]) / len(runs)
        print(
            f"  mean tokens/sec: {mean_tps:.1f}   mean TTFT: {mean_ttft * 1000:.0f}ms   VRAM: {vram_after}MB"
        )

### Cell 4 -- Summary table and chart

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

rows = []
for model, data in results.items():
    runs = data["runs"]
    rows.append(
        {
            "model": model,
            "mean_tokens_per_sec": sum(
                r["tokens_per_sec"] for r in runs if r["tokens_per_sec"]
            )
            / len(runs),
            "mean_ttft_ms": sum(r["ttft_s"] for r in runs if r["ttft_s"])
            / len(runs)
            * 1000,
            "vram_used_mb": data["vram_used_mb"],
        }
    )
df = pd.DataFrame(rows).sort_values("mean_tokens_per_sec", ascending=False)
display(df)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(df["model"], df["mean_tokens_per_sec"])
axes[0].set_title("Tokens/sec (higher is better)")
axes[0].tick_params(axis="x", rotation=30)
axes[1].bar(df["model"], df["mean_ttft_ms"], color="orange")
axes[1].set_title("Time to first token, ms (lower is better)")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

### Cell 5 -- Download the raw JSON

Includes a provenance header -- GPU model (from `nvidia-smi`), prompt set,
and sampling options -- so a number pulled out of this later still carries
what produced it, matching the discipline `backend/tools/measure/`'s reports
already follow (their `MeasurementReport.provenance` field exists for the
same reason).

In [ ]:
import datetime
import json as _json
import subprocess as _subprocess

try:
    gpu_name = _subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
except (_subprocess.CalledProcessError, FileNotFoundError):
    gpu_name = "UNKNOWN (no GPU / nvidia-smi unavailable)"

report = {
    "measurement_id": "colab_llm_throughput",
    "title": "Raw Ollama generation throughput across candidate model sizes (Colab)",
    "provenance": "live",
    "gpu": gpu_name,
    "prompts": PROMPTS,
    "num_predict": NUM_PREDICT,
    "measured_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "results": results,
    "note": (
        "Standalone raw-generation measurement, not one of "
        "backend/tools/measure/'s registered m11-m17 reports -- see this "
        "notebook's first cell for why."
    ),
}

out_path = "/content/llm_throughput_colab.json"
with open(out_path, "w") as f:
    _json.dump(report, f, indent=2)
print(f"Wrote {out_path}")

try:
    from google.colab import files

    files.download(out_path)
except ImportError:
    print("Not running in Colab -- find the file at", out_path)